# MMDS Cloud Pipeline — Google Colab
Run Phase 1/2 Bitcoin transaction preprocessing and graph features on a **small real Elliptic sample** using a free CPU Colab runtime. No training is included. Codex Cloud remains the development environment and GitHub holds code only.

**Execution status:** this notebook is shipped unexecuted. No Colab or real-data result is claimed. See [browser instructions](https://github.com/ojassgambira/MMDS/blob/main/docs/colab.md). Run cells individually from top to bottom; stop on errors. Never retry dataset downloads in Codex Cloud.

Select Runtime → Change runtime type → Python 3, CPU, a free runtime (no GPU, TPU or paid plan). Python must be 3.12 and Java 17/21. If Colab changes its default Python, select a compatible past runtime; do not remove the version guard. `local[2]` below refers to workers inside the Colab VM, never your Mac/iPhone.

In [ ]:
import sys
from pathlib import Path
import subprocess
import os
assert sys.version_info[:2] == (3, 12), 'Select a Colab Python 3.12 runtime'
assert Path('/content').exists(), 'Run this notebook in Google Colab only'
REPO_URL = 'https://github.com/ojassgambira/MMDS.git'
REPO_REF = 'main'  # Use the PR branch before merge; a full commit SHA freezes the code.
REPO = Path('/content/MMDS')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Use a clean checkout'
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(REPO)
COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repository commit:', COMMIT)

## Runtime dependencies
Install the existing pinned dependency closure plus pinned notebook plotting/Parquet tools. The notebook calls Spark in a fresh subprocess so dependency installation does not reuse an old Spark session. Java is supplied by Colab; this cell verifies its version. If installation fails, stop and inspect the error.

In [ ]:
import re
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-dev.lock'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-build-isolation', '--no-deps', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'matplotlib==3.10.6', 'pandas==2.2.3', 'pyarrow==20.0.0'], check=True)
java = subprocess.run(['java', '-version'], capture_output=True, text=True, check=True)
assert re.search(r'version "(17|21)\.', java.stderr), 'Java 17 or 21 is required'
print(java.stderr)
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['PYSPARK_PYTHON'] = sys.executable
# Editable installation happens after kernel startup; expose its source now.
sys.path.insert(0, str(REPO / 'src'))


## Storage and persistence
Raw CSVs, samples, Parquet and manifests stay outside the checkout. `/content/mmds` is temporary and disappears when Colab disconnects or resets. Set `PERSIST_TO_DRIVE=True` to mount your existing Google Drive and explicitly copy a completed run at the end. Spark writes to VM storage first because mounted Drive is slow and is not a distributed filesystem. Check existing Drive capacity; do not buy storage. Do not share raw/derived dataset artifacts publicly.

In [ ]:
import json
import shutil
from datetime import datetime, timezone
PERSIST_TO_DRIVE = False
ROOT = Path('/content/mmds')
ROOT.mkdir(exist_ok=True)
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RAW = ROOT / 'elliptic'
SAMPLE = ROOT / f'sample-{RUN_ID}'
OUTPUT = ROOT / f'run-{RUN_ID}'
DRIVE_ROOT = None
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/MMDS')
    DRIVE_ROOT.mkdir(exist_ok=True)
print('Temporary workspace:', ROOT, 'Free bytes:', shutil.disk_usage(ROOT).free)

## Authorized Elliptic acquisition
Publisher: [ellipticco/elliptic-data-set](https://www.kaggle.com/datasets/ellipticco/elliptic-data-set). The repository records **CC BY-NC-ND 4.0**; review current publisher terms for private noncommercial university analysis. The documented [PyG mirror](https://data.pyg.org/datasets/elliptic) provides the three original CSV archives without authentication. Dataset terms apply to the mirror too.

Default `pyg` uses the existing downloader: 200 MB total compressed / 500 MB expanded limits, redirect/path checks, hashes, and atomic publication. This acquires the source release before sampling; only the sample is processed by Spark. No multi-GB download is permitted. Failures stop the cell, with no automatic retries or source fallback.

For Kaggle, use its official dataset page in your browser, accept any terms, download the release and extract the three CSVs to a **private Drive folder**. Set `SOURCE_MODE='existing'` and `EXISTING_RAW` to that folder after mounting Drive. Do not upload data to GitHub.

If the official Kaggle client requires authentication, store your account's `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab's Secrets panel (key icon), grant this notebook access, and retrieve them with `google.colab.userdata.get` only in runtime memory. Pass them through the environment of the official client, then remove them. Never place literals, `kaggle.json`, tokens, passwords, or credential output in cells, Drive outputs, commits or screenshots. Authentication is unnecessary for the default PyG path. The notebook intentionally does not invoke an unbounded Kaggle CLI download.

In [ ]:
from mmd_fraud.data.acquire import acquire, FILES, sha256, outside_repo
SOURCE_MODE = 'pyg'  # 'existing' for publisher CSVs already in private Drive
EXISTING_RAW = Path('/content/drive/MyDrive/MMDS/elliptic')
if SOURCE_MODE == 'pyg':
    if RAW.exists():
        provenance = json.loads((RAW / 'acquisition.json').read_text())
        assert provenance['source'] == 'https://data.pyg.org/datasets/elliptic'
        records = {r['file']: r for r in provenance['files']}
        assert set(records) == set(FILES)
        for name in FILES:
            assert sha256(RAW / name) == records[name]['sha256'], 'Changed cached input'
        print('Reusing previously acquired, hash-verified source')
    else:
        provenance = acquire(RAW)
        print('Acquired source with manifest:', RAW / 'acquisition.json')
elif SOURCE_MODE == 'existing':
    assert PERSIST_TO_DRIVE, 'Mount Drive first'
    RAW = outside_repo(EXISTING_RAW)
    assert all((RAW / name).is_file() for name in FILES), 'Provide all three publisher CSVs'
    provenance = {'source': 'user-provided publisher CSVs; authorization checked by user',
                  'publisher': 'ellipticco/elliptic-data-set',
                  'files': [{'file': n, 'sha256': sha256(RAW / n)} for n in FILES]}
else:
    raise ValueError('Select pyg or existing')

## Source audit and deterministic real-data sample
Keep the first 40 transactions per source step and every edge whose endpoints are both selected (at most 1,960 nodes and 10,000 edge records). This induced subgraph is for correctness, not representative fraud rates or full-graph performance. It preserves source step values and unknown labels. Empty/oversized samples stop; adjust the bounded per-step value deliberately, never silently replace real data with fixtures.

The original source expectations are 203,769 nodes, 234,355 edge records, labels 4,545 illicit / 42,019 licit / 157,205 unknown, and steps 1–49. These assertions audit counts, not an official checksum or a full numeric validation of unsampled rows. Spark validates all sampled numeric values and relationships next.

In [ ]:
from mmd_fraud.data.sample import sample_elliptic
import time
sample_started = time.monotonic()
sample_manifest = sample_elliptic(RAW, SAMPLE, per_step=40)
sample_seconds = time.monotonic() - sample_started
assert sample_manifest['source_nodes'] == 203769
assert sample_manifest['source_edges_raw'] == 234355
assert sample_manifest['source_labels'] == {'1': 4545, '2': 42019, 'unknown': 157205}
assert sample_manifest['source_steps'] == list(range(1, 50))
(ROOT / f'provenance-{RUN_ID}.json').write_text(json.dumps(provenance, indent=2))
print(json.dumps(sample_manifest, indent=2))

## Run existing Phase 2 PySpark preprocessing and graph features
Use only the small profile and two exclusive historical cutoffs. `available_step=max(source_step,target_step)` and historical features use step **< cutoff**. Anonymous source features are retained for audit but never used as causal historical features. No labels enter graph computations. Parquet is round-trip checked by the existing runner; failures leave a truthful `run.json` and require a new run ID.

In [ ]:
config = {'directory': str(SAMPLE), 'missing_policy': 'reject',
          'require_original_counts': False, 'dataset_kind': 'real_elliptic_induced_sample'}
CONFIG = ROOT / f'config-{RUN_ID}.json'
CONFIG.write_text(json.dumps(config, indent=2))
subprocess.run([sys.executable, 'scripts/preprocess.py', '--config', str(CONFIG),
                '--profile', 'small', '--output', str(OUTPUT), '--master', 'local[2]',
                '--cutoffs', '2', '35'], check=True, timeout=720)
manifest = json.loads((OUTPUT / 'run.json').read_text())
assert manifest['status'] == 'success'
print(json.dumps(manifest['validation'], indent=2))

## Validate output counts, labels and time
This bounded driver read is limited to the small-profile outputs. Labels remain `1` illicit, `2` licit and `unknown`; unknown is never a negative. Historical degrees belong to the sampled graph only.

In [ ]:
import pandas as pd
nodes = pd.read_parquet(OUTPUT / 'nodes')
edges = pd.read_parquet(OUTPUT / 'edges')
validation = manifest['validation']
assert len(nodes) == validation['nodes'] == sample_manifest['sample_nodes']
assert len(edges) == validation['edges']
assert len(edges) + validation['edge_duplicates_removed'] == sample_manifest['sample_edges_raw']
assert nodes['label'].value_counts().to_dict() == validation['labels']
assert set(nodes['label']) <= {'1', '2', 'unknown'}
assert sorted(nodes['time_step'].astype(int).unique()) == validation['steps']
assert nodes['time_step'].astype(int).between(1, 49).all()
assert set(edges['source']) | set(edges['target']) <= set(nodes['entity_id'])
assert (edges['available_step'].astype(int) == edges[['source_step', 'target_step']].max(axis=1)).all()
for cutoff in (2, 35):
    historical = pd.read_parquet(OUTPUT / f'historical_features_{cutoff}')
    assert set(historical['entity_id']) == set(nodes.loc[nodes['time_step'].astype(int) < cutoff, 'entity_id'])
    assert (historical['time_step'].astype(int) < cutoff).all()
    assert (historical['cutoff_step'].astype(int) == cutoff).all()
print('Validated nodes:', len(nodes), 'edges:', len(edges), 'labels:', validation['labels'])

## Processing benchmarks
Report measured stage durations, rows and file counts from this run, not projected scalability. Acquisition/sample construction and pipeline time are separate. Small induced-graph timing cannot predict full release/cluster runtime. Python RSS is not JVM peak; cgroup readings may be unavailable or include earlier workloads. No training benchmark is included.

In [ ]:
from IPython.display import display
benchmark = pd.DataFrame.from_dict(manifest['stages'], orient='index')
display(benchmark)
print('Validation seconds:', manifest['validation_seconds'])
print('Pipeline elapsed seconds:', manifest['elapsed_seconds'])
benchmark.to_csv(OUTPUT / 'benchmark.csv', index_label='stage')
print('Sample construction seconds:', sample_seconds)
print('Acquisition seconds (if recorded):', provenance.get('elapsed_seconds', 'not recorded for existing CSVs'))
(OUTPUT / 'notebook-timings.json').write_text(json.dumps({'sample_seconds': sample_seconds, 'acquisition_seconds': provenance.get('elapsed_seconds')}))


## Visualize measured sample summaries
Plots describe the induced sample, not the full network or classifier performance. Elliptic steps are discrete source units, not calendar dates. Save figures beside run evidence.

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
nodes['label'].value_counts().reindex(['1', '2', 'unknown'], fill_value=0).plot.bar(ax=axes[0], title='Sample labels')
nodes['time_step'].astype(int).value_counts().sort_index().plot.bar(ax=axes[1], title='Nodes per Elliptic step')
graph = pd.read_parquet(OUTPUT / 'graph_features_descriptive')
graph['total_degree'].plot.hist(ax=axes[2], bins=20, title='Induced graph total degree')
fig.tight_layout()
fig.savefig(OUTPUT / 'sample-summary.png', dpi=150)
plt.show()

## Persist a completed run
Copy artifacts only if Drive was explicitly enabled. A unique destination prevents overwriting previous results. Copy sample/provenance/run evidence together. Raw source stays in temporary storage unless you deliberately archive it privately with its acquisition manifest; it can be re-acquired later. Verify Drive files before ending the runtime. Partially copied folders are not successful archives.

In [ ]:
if PERSIST_TO_DRIVE:
    destination = DRIVE_ROOT / RUN_ID
    assert not destination.exists(), 'Choose a new run ID'
    destination.mkdir()
    shutil.copytree(OUTPUT, destination / 'run')
    shutil.copytree(SAMPLE, destination / 'sample')
    shutil.copy2(CONFIG, destination / CONFIG.name)
    shutil.copy2(ROOT / f'provenance-{RUN_ID}.json', destination / 'provenance.json')
    for source_file in OUTPUT.rglob('*'):
        if source_file.is_file():
            assert sha256(source_file) == sha256(destination / 'run' / source_file.relative_to(OUTPUT))
    (destination / 'ARCHIVE_COMPLETE.json').write_text(json.dumps({'git_commit': COMMIT, 'status': 'copied_and_run_files_hash_verified'}))
    print('Private persistent archive:', destination)
else:
    print('Temporary outputs only:', OUTPUT, '— lost on runtime reset. Enable Drive and rerun this cell to persist.')

## Optional dataset-free checks
These checks use generated fixtures or mocked acquisition, never real downloads. The full distributed suite takes longer; no training runs. Development checks normally run in Codex Cloud/GitHub CI.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', 'test_colab.py', '-v'], check=True)
subprocess.run([sys.executable, '-m', 'ruff', 'check', '.'], check=True)